# H₀₄ — Retrieval-Stage Code-Mixing Penalty (+ retrieval baselines)

**Run this on Kaggle with a GPU.** It is the paper's headline experiment and it costs **zero API quota**.

It could not be run on the project laptop: LaBSE encodes at **0.38–0.46 texts/s** there
(memory-bound — larger batches are *slower*), so 9,045 texts is ~6 hours. On a Kaggle T4 it is minutes.

---

## What this tests

> **H₀₄:** retrieval quality does not differ between code-mixed Hinglish queries and semantically
> equivalent English renderings, holding encoder, index and relevance criterion constant.

Three query conditions over the **same** pairs and the **same** index:

| Variant | Query text | Meaning |
|---|---|---|
| **Q1** | `hinglish_query` | the deployed path |
| **Q2** | English question, caption stripped | translation ceiling |
| **Q3** | full English summary incl. caption | multimodal ceiling |

### ⚠️ Why Q2 must exist

MMCQSD's `english_summary` is a restated question **plus an image caption** of the form
`"The image here shows a medical condition related to swollen_tonsils."`
That caption contains the underscore-joined condition-group label — which **is the relevance label**
for this evaluation. Retrieving with the full summary therefore retrieves with the answer key.

So **Q1 vs Q2 is H₀₄**. Q3 is an upper bound, *not* an English baseline. The Q2→Q3 gap is a free bonus
result: it quantifies the headroom a perfect image reader would add.

### Setup

Create a Kaggle Dataset containing:
- `mmcqsd_multicare_paired.csv`
- `evidence_metadata.csv`
- `evidence.index` *(optional — rebuilt from metadata if absent)*

Attach it, then set `Settings → Accelerator → GPU`, and **Run All**.

In [ ]:
!pip -q install sentence-transformers faiss-cpu rank_bm25 2>/dev/null

import torch, platform
print("python :", platform.python_version())
print("torch  :", torch.__version__)
print("CUDA   :", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "(CPU only — this will be slow)")

In [ ]:
import glob, os, re, hashlib, json
from pathlib import Path

import numpy as np
import pandas as pd
import faiss
from scipy import stats

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# MUST match build_index.py, which sets encoder.model.max_seq_length = 128.
# Queries and documents encoded under different truncation are not comparable.
MAX_SEQ_LENGTH = 128
BATCH = 256 if DEVICE == "cuda" else 16
TOP_K = 10
K_VALUES = (1, 3, 5, 10)
N_BOOT = 10_000
SEED = 42

OUT = Path("/kaggle/working/h4_results"); OUT.mkdir(parents=True, exist_ok=True)

def find(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    if not hits:
        raise FileNotFoundError(f"{name} not found under /kaggle/input — attach the dataset.")
    return hits[0]

pairs = pd.read_csv(find("mmcqsd_multicare_paired.csv"))
meta  = pd.read_csv(find("evidence_metadata.csv"))
print(f"pairs {len(pairs)} | index metadata {len(meta)}")
print("pairs cols:", list(pairs.columns))
assert {"hinglish_query", "english_summary", "condition_query"} <= set(pairs.columns)
assert "condition_group" in meta.columns and "case_text" in meta.columns

## 1. Build Q1 / Q2 / Q3 and enforce the leakage gate

In [ ]:
CAPTION_RE = re.compile(r"\s*The image\b.*$", re.IGNORECASE | re.DOTALL)
LEAD_RE    = re.compile(r"^\s*summary\s*:\s*", re.IGNORECASE)

def strip_caption(s):
    return CAPTION_RE.sub("", LEAD_RE.sub("", str(s))).strip()

q1 = pairs["hinglish_query"].astype(str)
q3 = pairs["english_summary"].astype(str)
q2 = q3.apply(strip_caption)

def assert_no_leakage(q2, conditions):
    """Q2 must not contain the caption or the machine-readable group label.

    NOTE: we deliberately do NOT assert the absence of the condition's individual
    WORDS. A patient question legitimately describes its own symptom ("a lump in
    my neck") and 79.6% of stripped questions do. That is the query, not leakage.
    """
    n_cap = q2.str.contains(r"The image", case=False, regex=True).sum()
    assert n_cap == 0, f"{n_cap} rows still contain a caption"
    hits = sum(q2.str.contains(re.escape(l), case=False, regex=True).sum()
               for l in set(conditions.astype(str)) if "_" in l)
    assert hits == 0, f"{hits} rows still contain an underscore-joined label"
    assert (q2.str.len() == 0).sum() == 0, "empty rows after stripping"
    print("✓ leakage gate PASSED")

assert_no_leakage(q2, pairs["condition_query"])
print(f"mean chars  Q1={int(q1.str.len().mean())}  Q2={int(q2.str.len().mean())}  Q3={int(q3.str.len().mean())}")
print("\nexample:")
print("  Q2:", q2.iloc[0][:150])
print("  Q3:", q3.iloc[0][:150])

## 2. Metrics and paired tests

Relevance: a retrieved case is relevant iff its `condition_group` equals the query's.
Coarse (18 groups) but free and label-consistent — report as a limitation.

In [ ]:
def rank_metrics(hits):
    out = {f"recall@{k}": float(hits[:, :k].any(axis=1).mean()) for k in K_VALUES}
    first, has = np.argmax(hits, axis=1), hits.any(axis=1)
    rr = np.where(has, 1.0 / (first + 1), 0.0)
    out["MRR@10"] = float(rr.mean())
    disc = 1.0 / np.log2(np.arange(2, hits.shape[1] + 2))
    out["nDCG@10"] = float((hits * disc).sum(axis=1).clip(max=1.0).mean())
    return out

def reciprocal_ranks(hits):
    first = np.argmax(hits, axis=1)
    return np.where(hits.any(axis=1), 1.0 / (first + 1), 0.0)

def mcnemar(a, b):
    """Exact McNemar on paired binary outcomes."""
    a, b = a.astype(bool), b.astype(bool)
    n01, n10 = int((~a & b).sum()), int((a & ~b).sum())
    n = n01 + n10
    p = 1.0 if n == 0 else float(stats.binomtest(n10, n, 0.5).pvalue)
    return n01, n10, p

def bootstrap_delta(a, b, seed=SEED, n_boot=N_BOOT):
    """Percentile CI for mean(a)-mean(b) under PAIRED resampling."""
    rng = np.random.default_rng(seed); n = len(a)
    boot = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, n, n)
        boot[i] = a[idx].mean() - b[idx].mean()
    return float(np.percentile(boot, 2.5)), float(np.percentile(boot, 97.5))

def random_floor(query_conditions, meta):
    """Analytic prevalence-weighted floor. NOTE the index is condition-balanced
    while queries are not (skin_rash = 34.8% of queries vs 7.2% of the index), so
    the aggregate floor is dominated by a few conditions. Always read the
    per-condition table alongside the aggregate."""
    prev = meta["condition_group"].value_counts(normalize=True)
    p = query_conditions.map(prev).fillna(0.0).to_numpy()
    return {f"recall@{k}": float((1.0 - (1.0 - p) ** k).mean()) for k in K_VALUES}

gold = pairs["condition_query"].astype(str).to_numpy()
meta_cond = meta["condition_group"].to_numpy()
print("helpers ready")

## 3. LaBSE — the deployed encoder

Uses the shipped `evidence.index` when available, so this reproduces the deployed system exactly.

In [ ]:
from sentence_transformers import SentenceTransformer

def load_encoder(name):
    m = SentenceTransformer(name, device=DEVICE)
    m.max_seq_length = MAX_SEQ_LENGTH
    return m

def encode(model, texts, prefix=""):
    txt = [prefix + t for t in texts] if prefix else list(texts)
    return model.encode(txt, batch_size=BATCH, show_progress_bar=True,
                        normalize_embeddings=True, convert_to_numpy=True).astype(np.float32)

def build_index(emb):
    ix = faiss.IndexFlatIP(emb.shape[1]); ix.add(emb); return ix

labse = load_encoder("sentence-transformers/LaBSE")

try:
    index = faiss.read_index(find("evidence.index"))
    assert index.ntotal == len(meta), f"index {index.ntotal} != metadata {len(meta)}"
    print(f"✓ loaded shipped index ({index.ntotal} vectors)")
except FileNotFoundError:
    print("evidence.index not supplied — rebuilding from metadata (200-word truncation, as build_index.py)")
    docs = [" ".join(str(t).split()[:200]) for t in meta["case_text"]]
    index = build_index(encode(labse, docs))
    print(f"✓ rebuilt index ({index.ntotal} vectors)")

In [ ]:
variants = {"Q1_hinglish": q1, "Q2_english_question": q2, "Q3_english_plus_caption": q3}
hits, rows = {}, []

for name, texts in variants.items():
    print(f"\n--- {name} ---")
    emb = encode(labse, texts.tolist())
    _, idx = index.search(emb, TOP_K)
    h = meta_cond[idx] == gold[:, None]
    hits[name] = h
    rows.append({"system": "LaBSE", "variant": name, **rank_metrics(h)})
    np.save(OUT / f"emb_labse_{name}.npy", emb)

floor = random_floor(pairs["condition_query"], meta)
rows.append({"system": "random", "variant": "analytic_floor", **floor})
labse_metrics = pd.DataFrame(rows)
display(labse_metrics.round(4))

In [ ]:
comps = []
for lo, hi in [("Q1_hinglish", "Q2_english_question"),
               ("Q2_english_question", "Q3_english_plus_caption"),
               ("Q1_hinglish", "Q3_english_plus_caption")]:
    a1, b1 = hits[lo][:, 0], hits[hi][:, 0]
    n01, n10, p = mcnemar(a1, b1)
    rr_a, rr_b = reciprocal_ranks(hits[lo]), reciprocal_ranks(hits[hi])
    w = stats.wilcoxon(rr_a, rr_b, zero_method="zsplit")
    lo_ci, hi_ci = bootstrap_delta(b1.astype(float), a1.astype(float))
    comps.append({"comparison": f"{hi} - {lo}",
                  "delta_R@1": float(b1.mean() - a1.mean()),
                  "ci_lo": lo_ci, "ci_hi": hi_ci,
                  "n01": n01, "n10": n10, "mcnemar_p": p,
                  "wilcoxon_rr_p": float(w.pvalue),
                  "delta_MRR": float(rr_b.mean() - rr_a.mean())})
comps = pd.DataFrame(comps)
display(comps.round(5))
print("\nQ1 vs Q2 is H04 proper. Q3 is a ceiling, not a baseline.")

In [ ]:
# Per-condition breakdown — essential, because the index is condition-balanced
# while the query distribution is not.
per_cond = []
for cond in sorted(pairs["condition_query"].unique()):
    m = (pairs["condition_query"] == cond).to_numpy()
    row = {"condition": cond, "n_queries": int(m.sum()),
           "index_share": float((meta["condition_group"] == cond).mean())}
    for name in variants:
        row[f"{name}_R@1"] = float(hits[name][m, 0].mean())
    per_cond.append(row)
per_cond = pd.DataFrame(per_cond).sort_values("n_queries", ascending=False)
display(per_cond.round(4))

## 4. Encoder baselines

Each encoder needs **its own index** — you cannot search LaBSE vectors with e5 vectors.
Documents are re-encoded per model (10k docs; fast on GPU).

`multilingual-e5` requires the `query:` / `passage:` prefixes; omitting them materially hurts it.

In [ ]:
docs = [" ".join(str(t).split()[:200]) for t in meta["case_text"]]

ENCODERS = {
    "multilingual-e5-base": ("intfloat/multilingual-e5-base", "query: ", "passage: "),
    "MuRIL":                ("google/muril-base-cased",       "",        ""),
}

baseline_rows = []
for label, (model_id, qpre, dpre) in ENCODERS.items():
    print(f"\n===== {label} =====")
    try:
        enc = load_encoder(model_id)
        dix = build_index(encode(enc, docs, prefix=dpre))
        for name, texts in variants.items():
            _, idx = dix.search(encode(enc, texts.tolist(), prefix=qpre), TOP_K)
            h = meta_cond[idx] == gold[:, None]
            baseline_rows.append({"system": label, "variant": name, **rank_metrics(h)})
        del enc, dix
        torch.cuda.empty_cache() if DEVICE == "cuda" else None
    except Exception as e:
        print(f"!! {label} failed: {type(e).__name__}: {e}")

display(pd.DataFrame(baseline_rows).round(4))

In [ ]:
# Lexical baselines (CPU). BM25 should fail badly on Hinglish — that failure,
# quantified, is the motivation for cross-lingual embedding in the first place.
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

tok = lambda s: re.findall(r"[a-z0-9]+", str(s).lower())
bm25 = BM25Okapi([tok(d) for d in docs])

for name, texts in variants.items():
    top = np.vstack([np.argsort(bm25.get_scores(tok(t)))[::-1][:TOP_K] for t in texts])
    h = meta_cond[top] == gold[:, None]
    baseline_rows.append({"system": "BM25", "variant": name, **rank_metrics(h)})
    print(f"BM25 {name}: R@1={h[:,0].mean():.4f}")

vec = TfidfVectorizer(min_df=2, max_features=200_000, ngram_range=(1, 2))
D = vec.fit_transform(docs)
for name, texts in variants.items():
    S = linear_kernel(vec.transform(texts.tolist()), D)
    top = np.argsort(-S, axis=1)[:, :TOP_K]
    h = meta_cond[top] == gold[:, None]
    baseline_rows.append({"system": "TF-IDF", "variant": name, **rank_metrics(h)})
    print(f"TF-IDF {name}: R@1={h[:,0].mean():.4f}")

## 5. Save everything

In [ ]:
all_metrics = pd.concat([labse_metrics, pd.DataFrame(baseline_rows)], ignore_index=True)
all_metrics.to_csv(OUT / "h4_all_metrics.csv", index=False)
comps.to_csv(OUT / "h4_comparisons.csv", index=False)
per_cond.to_csv(OUT / "h4_per_condition.csv", index=False)

q1r = float(labse_metrics.query("variant=='Q1_hinglish'")["recall@1"].iloc[0])
q2r = float(labse_metrics.query("variant=='Q2_english_question'")["recall@1"].iloc[0])
q3r = float(labse_metrics.query("variant=='Q3_english_plus_caption'")["recall@1"].iloc[0])
fl  = floor["recall@1"]

summary = f"""# H04 Results (n = {len(pairs)})

Encoder LaBSE (max_seq_length={MAX_SEQ_LENGTH}), index {len(meta)} MultiCaRe cases, top-k {TOP_K}.
Relevance: retrieved condition_group == query condition_group. Device: {DEVICE}.

| Variant | R@1 |
|---|---:|
| Q1 Hinglish (deployed)   | {q1r:.4f} |
| Q2 English question      | {q2r:.4f} |
| Q3 English + caption     | {q3r:.4f} |
| random floor (analytic)  | {fl:.4f} |

- Deployed Hinglish retrieval is **{q1r/fl:.2f}x** the random floor.
- **H04 (Q1 vs Q2): {q2r-q1r:+.4f} absolute R@1** — the unconfounded code-mixing penalty.
- Q2 -> Q3 = {q3r-q2r:+.4f} — headroom a perfect image reader would add.
  Q3 is a CEILING (its caption contains the relevance label), never an English baseline.

See h4_all_metrics.csv, h4_comparisons.csv, h4_per_condition.csv.
"""
(OUT / "h4_summary.md").write_text(summary)
print(summary)
print("files:", [p.name for p in sorted(OUT.iterdir())])
print("\nDownload /kaggle/working/h4_results and commit it to results/h4_retrieval/")